# Hybrid document search: taxonomy, ontology, and knowledge graphs

**Guide:** [`071-hybrid-document-search.md`](../guides/concepts/071-hybrid-document-search.md)  
**Prerequisite:** [`070-semantic-document-search.ipynb`](070-semantic-document-search.ipynb)

This end-to-end toy system combines lexical BM25, semantic embeddings indexed by FAISS, and knowledge-graph evidence. A taxonomy classifies and filters document chunks; ontology-style rules validate graph facts; the graph links facts to source chunks; reciprocal-rank fusion combines the retrieval lists. These parts complement one another: the taxonomy organizes, the ontology defines valid meaning, the graph stores instance facts, and the retrieval methods find supporting passages.

The sample corpus and graph facts are defined in this notebook so the non-model cells run offline. The embedding cell downloads `all-MiniLM-L6-v2` on first use and needs internet access. Install the project notebook dependencies from the repository root with `python -m pip install -e ".[notebooks]"`.

## 1. Prepare document chunks

Each chunk includes a stable identifier, category, source name, page, and text. Keeping these fields beside the text lets search filters and results preserve provenance.

In [ ]:
chunks = [
    {"id": "chunk-1", "source": "device-inventory.pdf", "page": 1, "category": "DeviceDocs", "text": "Device-D1 contains lithium battery Battery-B7, a field-replaceable component installed in the device."},
    {"id": "chunk-2", "source": "device-inventory.pdf", "page": 2, "category": "DeviceDocs", "text": "Device-D1 runs firmware release Firmware-2.4 for sensor readings and device control."},
    {"id": "chunk-3", "source": "security-bulletin.pdf", "page": 3, "category": "SecurityDocs", "text": "Advisory-A9 affects Firmware-2.4. Install the security update on devices using this release."},
    {"id": "chunk-4", "source": "supplier-records.txt", "page": 1, "category": "SupplierDocs", "text": "Battery-B7 is manufactured by Maker-M3 under the approved supplier agreement."},
    {"id": "chunk-5", "source": "maintenance-manual.txt", "page": 5, "category": "MaintenanceDocs", "text": "Before replacing a battery, shut down the device and follow the service checklist."},
    {"id": "chunk-6", "source": "security-bulletin.pdf", "page": 4, "category": "SecurityDocs", "text": "Security advisories explain affected software versions, mitigations, and upgrade guidance."},
]
chunk_by_id = {chunk["id"]: chunk for chunk in chunks}
print(f"Prepared {len(chunks)} source-linked document chunks")

## 2. Use a taxonomy to classify and filter

A taxonomy is a hierarchy of categories. The filter below includes every descendant of the requested category, so selecting `ProductDocuments` also includes its `DeviceDocs` and `SecurityDocs` children.

In [ ]:
taxonomy_parent = {
    "AllDocuments": None,
    "ProductDocuments": "AllDocuments",
    "DeviceDocs": "ProductDocuments",
    "SecurityDocs": "ProductDocuments",
    "SupplierDocs": "AllDocuments",
    "MaintenanceDocs": "AllDocuments",
}

def category_is_a(category, requested_parent):
    """Return whether category is the requested category or one of its descendants."""
    current = category
    while current is not None:
        if current == requested_parent:
            return True
        current = taxonomy_parent.get(current)
    return False

def filter_chunks(category=None):
    """Apply a taxonomy filter consistently to all retrieval methods."""
    if category is None:
        return chunks
    if category not in taxonomy_parent:
        raise ValueError(f"Unknown taxonomy category: {category}")
    return [chunk for chunk in chunks if category_is_a(chunk["category"], category)]

print("Product-document categories:", sorted({c["category"] for c in filter_chunks("ProductDocuments")}))

## 3. Use ontology-style rules to validate graph facts

The ontology declares entity classes and allowed relation domain/range types. This small validator checks the declarations; it is not a full logical reasoner.

In [ ]:
class_parent = {
    "Entity": None,
    "Product": "Entity",
    "Device": "Product",
    "Component": "Product",
    "FirmwareRelease": "Product",
    "Organization": "Entity",
    "Manufacturer": "Organization",
    "Advisory": "Entity",
}
entity_types = {
    "Device-D1": "Device",
    "Battery-B7": "Component",
    "Firmware-2.4": "FirmwareRelease",
    "Advisory-A9": "Advisory",
    "Maker-M3": "Manufacturer",
}
relation_rules = {
    "installed_in": ("Component", "Device"),
    "uses_firmware": ("Device", "FirmwareRelease"),
    "affected_by": ("FirmwareRelease", "Advisory"),
    "manufactured_by": ("Component", "Manufacturer"),
}

def class_is_a(actual, expected):
    """Check class inheritance used by ontology domain and range rules."""
    current = actual
    while current is not None:
        if current == expected:
            return True
        current = class_parent.get(current)
    return False

def validate_fact(fact):
    """Accept only known entities connected by a relation with compatible types."""
    rule = relation_rules.get(fact["relation"])
    if rule is None or fact["subject"] not in entity_types or fact["object"] not in entity_types:
        return False
    return (
        class_is_a(entity_types[fact["subject"]], rule[0])
        and class_is_a(entity_types[fact["object"]], rule[1])
    )

## 4. Build a knowledge graph with evidence provenance

Each fact is a subject-relation-object edge and points to the chunk that supports it. Validation prevents invalidly typed facts from entering the graph.

In [ ]:
facts = [
    {"subject": "Battery-B7", "relation": "installed_in", "object": "Device-D1", "chunk_id": "chunk-1"},
    {"subject": "Device-D1", "relation": "uses_firmware", "object": "Firmware-2.4", "chunk_id": "chunk-2"},
    {"subject": "Firmware-2.4", "relation": "affected_by", "object": "Advisory-A9", "chunk_id": "chunk-3"},
    {"subject": "Battery-B7", "relation": "manufactured_by", "object": "Maker-M3", "chunk_id": "chunk-4"},
]
invalid_fact = {"subject": "Device-D1", "relation": "installed_in", "object": "Battery-B7", "chunk_id": "chunk-1"}
assert all(validate_fact(fact) for fact in facts)
assert not validate_fact(invalid_fact)
print(f"Validated {len(facts)} graph facts; rejected an invalid relation")

def graph_search(seed_entity, max_hops=2):
    """Find evidence chunks connected to an entity by traversing validated edges."""
    if seed_entity not in entity_types:
        raise ValueError(f"Unknown graph entity: {seed_entity}")
    if max_hops < 0:
        raise ValueError("max_hops must be non-negative")
    frontier = {seed_entity}
    visited = set(frontier)
    evidence_ids = []
    for _ in range(max_hops):
        next_frontier = set()
        for fact in facts:
            if not validate_fact(fact):
                continue
            if fact["subject"] in frontier or fact["object"] in frontier:
                evidence_ids.append(fact["chunk_id"])
                next_frontier.update({fact["subject"], fact["object"]} - visited)
        visited.update(next_frontier)
        frontier = next_frontier
    return list(dict.fromkeys(evidence_ids))

print("Evidence linked to Device-D1:", graph_search("Device-D1"))

## 5. Retrieve with lexical BM25

BM25 scores exact term overlap while accounting for term frequency, document frequency, and chunk length. It is particularly helpful for product IDs and exact advisory names.

In [ ]:
import math
import re
from collections import Counter


def tokenize(text):
    """Normalize text into simple alphanumeric search terms."""
    return re.findall(r"\w+", text.casefold())

def bm25_search(query, candidate_chunks, k1=1.5, b=0.75):
    """Return chunk IDs ordered by BM25 score, with stable ties."""
    query_terms = tokenize(query)
    if not query_terms or not candidate_chunks:
        return []
    tokenized = {chunk["id"]: tokenize(chunk["text"]) for chunk in candidate_chunks}
    average_length = sum(map(len, tokenized.values())) / len(tokenized)
    document_frequency = Counter(
        term for terms in tokenized.values() for term in set(terms)
    )
    scored = []
    total = len(tokenized)
    for chunk in candidate_chunks:
        terms = tokenized[chunk["id"]]
        frequencies = Counter(terms)
        length = len(terms)
        score = 0.0
        for term in query_terms:
            frequency = frequencies[term]
            if frequency == 0:
                continue
            idf = math.log(1 + (total - document_frequency[term] + 0.5) / (document_frequency[term] + 0.5))
            denominator = frequency + k1 * (1 - b + b * length / max(average_length, 1))
            score += idf * frequency * (k1 + 1) / denominator
        if score > 0:
            scored.append((chunk["id"], score))
    return [chunk_id for chunk_id, _ in sorted(scored, key=lambda item: (-item[1], item[0]))]

query = "Which security advisory affects the firmware on Device-D1?"
lexical_ranking = bm25_search(query, filter_chunks("ProductDocuments"))
print("BM25 result order:", lexical_ranking)

## 6. Retrieve semantically with embeddings and FAISS

Like lesson 070, unit-normalized embeddings searched with FAISS inner product approximate cosine similarity. This model-loading cell downloads a pretrained model on first run.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedding_model.encode(
    [chunk["text"] for chunk in chunks],
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")
faiss.normalize_L2(chunk_vectors)
vector_index = faiss.IndexFlatIP(chunk_vectors.shape[1])
vector_index.add(chunk_vectors)
print(f"FAISS contains {vector_index.ntotal} document vectors")

## 7. Fuse independent rankings with RRF

RRF uses positions in each list instead of adding incomparable raw BM25 and cosine scores. A graph result can vote as a third ranked list.

In [ ]:
def reciprocal_rank_fusion(rankings, rank_constant=60):
    """Fuse ranked ID lists, counting an ID at most once in each list."""
    if rank_constant <= 0:
        raise ValueError("rank_constant must be positive")
    scores = Counter()
    for ranking in rankings:
        for rank, item_id in enumerate(dict.fromkeys(ranking), start=1):
            scores[item_id] += 1 / (rank_constant + rank)
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))

graph_ranking = graph_search("Device-D1")
fused_demo = reciprocal_rank_fusion([lexical_ranking, graph_ranking])
print("Lexical plus graph RRF:", fused_demo)
assert fused_demo[0][0] in set(lexical_ranking + graph_ranking)

## 8. Run the complete hybrid search

A category filter is applied before each retrieval method, so lexical, vector, and graph candidates obey the same taxonomy boundary. The returned `matched_by` list makes each chunk's retrieval evidence visible.

In [ ]:
def hybrid_search(query, category=None, seed_entity=None, top_k=5):
    """Combine taxonomy-filtered BM25, FAISS, and graph candidate rankings."""
    if not query.strip():
        raise ValueError("query must not be empty")
    if top_k <= 0:
        raise ValueError("top_k must be positive")
    candidates = filter_chunks(category)
    if not candidates:
        return []
    allowed_ids = {chunk["id"] for chunk in candidates}
    lexical = bm25_search(query, candidates)
    query_vector = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vector)
    _scores, positions = vector_index.search(query_vector, len(chunks))
    semantic = [
        chunks[position]["id"]
        for position in positions[0]
        if position >= 0 and chunks[position]["id"] in allowed_ids
    ]
    graph = [chunk_id for chunk_id in graph_search(seed_entity) if chunk_id in allowed_ids] if seed_entity else []
    rankings = {"bm25": lexical, "semantic": semantic, "knowledge_graph": graph}
    fused = reciprocal_rank_fusion(list(rankings.values()))[:top_k]
    results = []
    for chunk_id, rrf_score in fused:
        chunk = chunk_by_id[chunk_id]
        results.append({
            **chunk,
            "rrf_score": rrf_score,
            "matched_by": [name for name, ranking in rankings.items() if chunk_id in ranking],
        })
    return results

In [ ]:
results = hybrid_search(
    "Which security advisory affects the firmware on Device-D1?",
    category="ProductDocuments",
    seed_entity="Device-D1",
    top_k=5,
)
for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['source']} page {result['page']} | {result['matched_by']} | RRF={result['rrf_score']:.4f}")
    print("   " + result["text"])
assert results and all(result["category"] in {"DeviceDocs", "SecurityDocs"} for result in results)

## What the components contributed

- The **taxonomy** restricted retrieval to product documentation and its descendants.
- The **ontology** checked that graph facts used appropriate entity and relation types.
- The **knowledge graph** connected a device to its firmware and advisory, while preserving source chunk IDs.
- **BM25** supplied exact-term candidates; **FAISS** supplied embedding-similarity candidates.
- **RRF** produced one ordering and `matched_by` explains each result's retrieval paths.

This small example does not extract facts automatically, implement ontology reasoning, persist the index, enforce user authorization, or generate answers. Verify every cited chunk against its original document before using retrieved evidence.